In [17]:
#1
import sys, re, gc, json, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
from nltk.stem import PorterStemmer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ROOT = Path("../..").resolve()          # project root (adjust if your notebook sits elsewhere)
sys.path.insert(0, str(ROOT))
from factchecking_main import read_passages, read_fact_examples

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

CACHE_DIR = Path("cache"); CACHE_DIR.mkdir(exist_ok=True)

In [18]:
#2
fact_to_passage = read_passages(str(ROOT / "data/passages_bm25_ChatGPT_humfacts.jsonl"))
examples = read_fact_examples(str(ROOT / "data/dev_labeled_ChatGPT.jsonl"), fact_to_passage)

y_true = np.array([1 if ex.label == "S" else 0 for ex in examples])   # IR counts as NS
N_FACTS = len(examples)
print(N_FACTS, "facts | S =", y_true.sum(), "| NS/IR =", (y_true == 0).sum())

221 facts | S = 112 | NS/IR = 109


In [19]:
#3
MODEL_NAME = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()
print(model.config.id2label)   # expect {0: entailment, 1: neutral, 2: contradiction}

def entailment_probs(premises, hypotheses, batch_size=16, max_length=512):
    """Returns an (n, 3) array of [P(entail), P(neutral), P(contradict)]."""
    out = []
    with torch.no_grad():                                   # no gradients: saves memory
        for s in range(0, len(premises), batch_size):
            enc = tokenizer(premises[s:s+batch_size], hypotheses[s:s+batch_size],
                            padding=True, truncation=True, max_length=max_length,
                            return_tensors="pt")
            out.append(torch.softmax(model(**enc).logits, dim=-1).numpy())
            del enc
    gc.collect()
    return np.vstack(out)

# Smoke test: should print E, C, N in that order
tests = [("Lanny Flaherty (born July 27, 1942) is an American actor.", "Lanny Flaherty is an American."),
         ("Lanny Flaherty (born July 27, 1942) is an American actor.", "Lanny Flaherty was born in 1950."),
         ("Lanny Flaherty (born July 27, 1942) is an American actor.", "Lanny Flaherty plays the guitar.")]
print(np.round(entailment_probs([p for p, _ in tests], [h for _, h in tests]), 3))

{0: 'entailment', 1: 'neutral', 2: 'contradiction'}
[[0.998 0.002 0.   ]
 [0.    0.    0.999]
 [0.    0.997 0.003]]


In [20]:
#4
stemmer = PorterStemmer()
STOP = set("""a an the and or but if then else when while is am are was were be been being
to of in on at by for from with as it its this that these those he she they them their we our
you your i me my""".split())          # negations (not, no, never) deliberately kept

def toks(text):
    text = re.sub(r"<[^>]+>", " ", text.lower())
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return {stemmer.stem(t) for t in text.split() if t not in STOP}

def jaccard(a, b):
    return len(a & b) / len(a | b) if a and b else 0.0

In [21]:
#5
def split_passage(text):
    sents = []
    for chunk in re.split(r"</?s>", text):
        chunk = chunk.strip()
        if not chunk: continue
        for s in re.split(r"(?<=[.!?])\s+", chunk):
            s = re.sub(r"\s+", " ", s).strip()
            if len(s.split()) >= 3:            # drop fragments like "Career."
                sents.append(s)
    return sents

KMAX = 10_000   # we run NLI on at most the top-8 sentences per fact

rows = []                 # one row per (fact, retrieved sentence)
fact_jacc = np.zeros(N_FACTS)   # whole-passage Jaccard per fact (used for pruning)

for i, ex in enumerate(examples):
    f = toks(ex.fact)
    cands = []
    best_passage = 0.0
    for p in ex.passages:
        best_passage = max(best_passage, jaccard(f, toks(p["text"])))
        sents = split_passage(p["text"])
        for j, s in enumerate(sents):
            cands.append({"fact_idx": i, "premise": s,
                          "premise_window": (sents[j-1] + " " + s) if j > 0 else s,
                          "jacc": jaccard(f, toks(s))})
    fact_jacc[i] = best_passage
    cands.sort(key=lambda r: r["jacc"], reverse=True)     # best Jaccard first
    for rank, r in enumerate(cands[:KMAX]):
        r["rank"] = rank
        rows.append(r)

print(len(rows), "pairs to run through the NLI model")

5417 pairs to run through the NLI model


In [22]:
#6
hyps = [examples[r["fact_idx"]].fact for r in rows]

t0 = time.time()
probs_sent   = entailment_probs([r["premise"] for r in rows], hyps)
probs_window = entailment_probs([r["premise_window"] for r in rows], hyps)
print(f"NLI done in {time.time()-t0:.0f}s")

np.savez(CACHE_DIR / "tamadhi_probs.npz",
         fact_idx=np.array([r["fact_idx"] for r in rows]),
         rank=np.array([r["rank"] for r in rows]),
         jacc=np.array([r["jacc"] for r in rows]),
         probs_sent=probs_sent, probs_window=probs_window,
         fact_jacc=fact_jacc, y_true=y_true)

NLI done in 944s


In [30]:
#7
fi   = np.array([r["fact_idx"] for r in rows])
rk   = np.array([r["rank"] for r in rows])
by_fact = defaultdict(list)
for idx, f in enumerate(fi): by_fact[f].append(idx)
by_fact = {f: np.array(v) for f, v in by_fact.items()}

PRUNED = -2.0                       # score for pruned facts, below any possible threshold
GRID = np.linspace(-1, 1, 2001)

def fact_scores(P, K, rule):
    """Aggregate sentence-level NLI outputs into one score per fact, using the top-K Jaccard sentences."""
    s = np.zeros(N_FACTS)
    for i in range(N_FACTS):
        idx = by_fact.get(i, np.array([], dtype=int))
        idx = idx[rk[idx] < K]
        if len(idx) == 0: s[i] = 0.0; continue
        if rule == "max_PE":
            s[i] = P[idx, 0].max()
        elif rule == "top2_PE":
            s[i] = np.sort(P[idx, 0])[::-1][:2].mean()
        elif rule == "margin":
            s[i] = (P[idx, 0] - P[idx, 2]).max()
    return s

def acc(pred): return float((pred == y_true).mean())

def tune(scores):
    accs = np.array([acc((scores >= t).astype(int)) for t in GRID])
    b = int(np.argmax(accs))
    return float(GRID[b]), float(accs[b])

def confusion(pred):
    return dict(TP=int(((pred==1)&(y_true==1)).sum()), FN=int(((pred==0)&(y_true==1)).sum()),
                FP=int(((pred==1)&(y_true==0)).sum()), TN=int(((pred==0)&(y_true==0)).sum()))

In [31]:
#8
results = []
for pname, P in [("sentence", probs_sent), ("window", probs_window)]:
    for K in [1, 3, 5, 8]:
        for rule in ["max_PE", "top2_PE", "margin"]:
            th, a = tune(fact_scores(P, K, rule))
            results.append((pname, K, rule, th, a))

results.sort(key=lambda r: r[4], reverse=True)
print(f"{'premise':<10}{'K':>3}  {'rule':<9}{'theta':>8}{'acc':>8}")
for r in results[:12]:
    print(f"{r[0]:<10}{r[1]:>3}  {r[2]:<9}{r[3]:>8.3f}{r[4]:>8.4f}")

premise     K  rule        theta     acc
window      5  max_PE      0.225  0.8190
window      5  top2_PE     0.260  0.8190
window      5  margin      0.216  0.8190
window      8  max_PE      0.225  0.8145
window      8  margin      0.216  0.8145
window      8  top2_PE     0.260  0.8100
sentence    5  top2_PE     0.218  0.8054
sentence    8  max_PE      0.322  0.8054
sentence    8  top2_PE     0.176  0.8054
window      3  max_PE      0.332  0.8054
sentence    5  max_PE      0.322  0.8009
sentence    8  margin      0.283  0.8009


In [32]:
#9
best_cfg = results[0]
PREM, K_BEST, RULE = best_cfg[0], best_cfg[1], best_cfg[2]
P_BEST = probs_window if PREM == "window" else probs_sent
base = fact_scores(P_BEST, K_BEST, RULE)
print("Using:", PREM, "K =", K_BEST, RULE)

pairs_per_fact = np.array([len(by_fact.get(i, [])) for i in range(N_FACTS)])
print(f"{'tau':>6}{'pruned':>8}{'S lost':>8}{'calls skipped':>15}{'theta':>8}{'acc':>8}")
for tau in [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.07, 0.10]:
    keep = fact_jacc >= tau
    sc = np.where(keep, base, PRUNED)
    th, a = tune(sc)
    skipped = pairs_per_fact[~keep].sum() / pairs_per_fact.sum()
    print(f"{tau:>6.2f}{(~keep).sum():>8}{int((~keep & (y_true==1)).sum()):>8}{skipped:>15.1%}{th:>8.3f}{a:>8.4f}")

Using: window K = 5 max_PE
   tau  pruned  S lost  calls skipped   theta     acc
  0.00       0       0           0.0%   0.225  0.8190
  0.01      13       1           4.3%   0.225  0.8190
  0.02      39       2          12.6%   0.225  0.8235
  0.03      89      24          37.4%   0.225  0.8009
  0.04     147      61          62.6%   0.225  0.6697
  0.05     167      76          72.3%   0.019  0.6154
  0.07     204     102          90.8%   0.990  0.5249
  0.10     217     110          98.2%   0.001  0.4977


In [36]:
# Pruning signal = best single-sentence Jaccard (far less diluted than whole-passage)
jacc_arr = np.array([r["jacc"] for r in rows])
fact_jacc_sent = np.zeros(N_FACTS)
for i in range(N_FACTS):
    idx = by_fact.get(i, np.array([], dtype=int))
    fact_jacc_sent[i] = jacc_arr[idx].max() if len(idx) else 0.0

# Use the best no-pruning setting: all sentences
K_FINAL = 10_000
base_full = fact_scores(P_BEST, K_FINAL, RULE)
print("Using:", PREM, "K =", K_FINAL, RULE)

print(f"{'tau':>6}{'pruned':>8}{'S lost':>8}{'calls skipped':>15}{'theta':>8}{'acc':>8}")
for tau in [0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.15, 0.20]:
    keep_s = fact_jacc_sent >= tau
    sc = np.where(keep_s, base_full, PRUNED)
    th, a = tune(sc)
    skipped = pairs_per_fact[~keep_s].sum() / pairs_per_fact.sum()
    print(f"{tau:>6.2f}{(~keep_s).sum():>8}{int((~keep_s & (y_true==1)).sum()):>8}"
          f"{skipped:>15.1%}{th:>8.3f}{a:>8.4f}")

Using: window K = 10000 max_PE
   tau  pruned  S lost  calls skipped   theta     acc
  0.00       0       0           0.0%   0.314  0.8416
  0.02      10       1           2.7%   0.314  0.8416
  0.04      10       1           2.7%   0.314  0.8416
  0.06      14       1           4.9%   0.314  0.8416
  0.08      25       4           8.4%   0.314  0.8371
  0.10      42      10          15.2%   0.314  0.8281
  0.12      58      17          21.4%   0.164  0.8009
  0.15      99      34          39.0%   0.045  0.7602
  0.20     139      55          58.2%   0.225  0.7149


In [45]:
#10
TAU = 0.06                       # <- choose from the Cell 9b table
K_BEST = K_FINAL
keep = fact_jacc_sent >= TAU
final_scores = np.where(keep, base_full, PRUNED)
THETA, final_acc = tune(final_scores)
pred = (final_scores >= THETA).astype(int)
print(f"tau={TAU} theta={THETA:.3f} acc={final_acc:.4f}", confusion(pred))

# plateau of best theta, and range where acc >= 83%
accs = np.array([acc((final_scores >= t).astype(int)) for t in GRID])
print("best plateau:", GRID[accs == accs.max()].min(), "to", GRID[accs == accs.max()].max())
print(f"final acc = {final_acc:.4f} (tau={TAU}, theta={THETA:.3f})")
ok = GRID[accs >= 0.83]
print(">=83% range:", (ok.min(), ok.max()) if len(ok) else "never reached")

# 5-fold CV: tune theta on 4 folds, test on the 5th
rng = np.random.default_rng(SEED)
order = rng.permutation(N_FACTS)
cv = []
for k, test in enumerate(np.array_split(order, 5), 1):
    train = np.setdiff1d(order, test)
    tr_accs = [((final_scores[train] >= t).astype(int) == y_true[train]).mean() for t in GRID]
    th = GRID[int(np.argmax(tr_accs))]
    ho = ((final_scores[test] >= th).astype(int) == y_true[test]).mean()
    cv.append(ho); print(f"fold {k}: theta={th:.3f} acc={ho:.3f}")
print(f"CV mean {np.mean(cv):.3f} ± {np.std(cv):.3f}")

tau=0.06 theta=0.314 acc=0.8416 {'TP': 96, 'FN': 16, 'FP': 19, 'TN': 90}
best plateau: 0.31400000000000006 to 0.3360000000000001
final acc = 0.8416 (tau=0.06, theta=0.314)
>=83% range: (np.float64(0.19900000000000007), np.float64(0.44399999999999995))
fold 1: theta=0.314 acc=0.844
fold 2: theta=0.199 acc=0.818
fold 3: theta=0.314 acc=0.841
fold 4: theta=0.225 acc=0.795
fold 5: theta=0.314 acc=0.841
CV mean 0.828 ± 0.019


In [38]:
#11
c = confusion(pred)
pS  = c["TP"]/(c["TP"]+c["FP"]); rS = c["TP"]/(c["TP"]+c["FN"])
pNS = c["TN"]/(c["TN"]+c["FN"]); rNS = c["TN"]/(c["TN"]+c["FP"])
print(f"S : P={pS:.3f} R={rS:.3f} F1={2*pS*rS/(pS+rS):.3f}")
print(f"NS: P={pNS:.3f} R={rNS:.3f} F1={2*pNS*rNS/(pNS+rNS):.3f}")

errors = [{"type": "FP" if pred[i] == 1 else "FN", "fact": ex.fact, "gold": ex.label,
           "score": round(float(final_scores[i]), 4), "pruned": bool(not keep[i])}
          for i, ex in enumerate(examples) if pred[i] != y_true[i]]
json.dump(errors, open("tamadhi_entailment_errors.json", "w"), indent=2)
print(len(errors), "errors saved")

S : P=0.839 R=0.839 F1=0.839
NS: P=0.835 R=0.835 F1=0.835
36 errors saved


In [43]:
#12
class EntailmentFactChecker(object):
    def __init__(self, ent_model, tau=0.03, theta=0.30, k=5, batch_size=16):
        self.ent_model = ent_model
        self.tau, self.theta, self.k, self.bs = tau, theta, k, batch_size

    def predict(self, fact, passages):
        f = toks(fact)
        cands = []
        for p in passages:
            sents = split_passage(p["text"])
            for j, s in enumerate(sents):
                cands.append((jaccard(f, toks(s)), (sents[j-1] + " " + s) if j > 0 else s))
        if not cands or max(c[0] for c in cands) < self.tau:
            return "NS"
        cands.sort(key=lambda x: x[0], reverse=True)
        premises = [c[1] for c in cands[:self.k]]

    def _p_entail(self, premise, hypothesis):
        # Adapt to your starter's EntailmentModel.check_entailment (may return logits)
        out = self.ent_model.check_entailment(premise, hypothesis)
        out = torch.softmax(torch.as_tensor(out, dtype=torch.float32).flatten(), dim=-1)
        return float(out[0])      # index 0 = entailment for this model

In [44]:
fn = np.where((pred == 0) & (y_true == 1))[0]
print("FN total:", len(fn))
print("  of which pruned by tau:", int((~keep[fn]).sum()))
print("  not pruned, model score low:", int(keep[fn].sum()))

# what would accuracy be with no pruning and every sentence checked?
for K in [K_BEST, 20, 10_000]:
    th, a = tune(fact_scores(P_BEST, K, RULE))
    print(f"no pruning, K={K}: theta={th:.3f} acc={a:.4f}")

FN total: 18
  of which pruned by tau: 4
  not pruned, model score low: 14
no pruning, K=10000: theta=0.314 acc=0.8416
no pruning, K=20: theta=0.225 acc=0.8235
no pruning, K=10000: theta=0.314 acc=0.8416
